# Build a legal document conflict detector

_Spotting deviations in contracts, laws, and policies_

This notebook is the JavaScript version of [the tutorial](https://isaacus.com/blog/build-a-legal-document-conflict-detector): the full text, with every code block as a cell.

It runs on the [Deno](https://docs.deno.com/runtime/reference/cli/jupyter/) kernel: install it once with `deno jupyter --install`.
Start Jupyter from this folder, for example with `uvx --from jupyterlab jupyter-lab conflict_detector_js.ipynb`. `deno.json` here tells Deno which npm packages the code imports, and Deno downloads them itself.
It reads `.env` in `source_files/js/` and the documents in `source_files/samples/`, and writes to `source_files/js/results/`.
The cells that run a test call the models, so they cost a few credits each time they run.

_This tutorial is aimed at intermediate Python or JavaScript users who understand LLMs. Try our [Build a legal RAG app that won't be held in contempt](https://isaacus.com/blog/build-a-legal-rag-app-that-wont-be-held-in-contempt) guide for something simpler._

## tl;dr

This guide walks through building an app that finds conflicts between two legal documents: provisions that conflict with each other, and provisions one document has that the other doesn't.

Every finding is grounded in the text of both documents. The crux of the work is handled by Isaacus' [enrichment](https://isaacus.com/docs/models/introduction#enrichment), [embedding](https://isaacus.com/docs/models/introduction#embedding) and [universal classification](https://isaacus.com/docs/models/introduction#universal-classification) models. A generative model is used to summarize **findings**.

While we focus on the backend, the entire demo app below is available for download in our [open-source cookbook GitHub repo](https://github.com/isaacus-dev/cookbooks).
_The interactive demo is in [the post](https://isaacus.com/blog/build-a-legal-document-conflict-detector)._

## The problem: finding conflict between provisions

Comparisons have been a part of legal work for a long time: manually reading line by line, while tracking a checklist or a playbook, all to create issue lists for review. Extraction and redlining grew easier with tools like Microsoft Word's change tracking, Kira, and contract lifecycle management platforms... but that was still fairly manual.

And slow.

Then gen AI emerged, with all sorts of models being integrated into legal review tooling. They provided some speed on the document processing side – great – but came with an Achilles' heel: recall. Point a generative model at two documents and ask for conflicts: it'll find some, but it won't do a comprehensive pairwise comparison, it doesn't scale to long documents, and it can miss obvious conflicts. Hallucinations are still a problem on top of that.

When introducing automation, there are a few things to take into consideration, which include:

- What constitutes a difference between two provisions in separate documents?
- How large is the volume of differences?
- Which of the differences found are actually significant?
- How can human understanding of the differences become more efficient?

We can take these questions into consideration while improving recall and precision by relegating gen AI models to what they're actually good at: summarizing and editing, rather than reviewing. For reviewing, we're going to replace the void they left behind with models that don't hallucinate: [Isaacus' models](https://isaacus.com/docs/models/introduction). The generative model only comes in at the very end, so any hallucinations are confined to the summaries, recommendations, and rewrites.

_Side note: during this article's drafting, I was fresh out of an Odyssey IMAX screening. Every time I mention 'provisions', know I'm thinking of the provisions Odysseus would've had packed for a voyage. And probably hungry._

## Process

Fundamentally, we have an incoming **input document** we want to compare to an existing **reference document** we already have. We divide them up into provisions, and work through each pair making decisions about them. We call all the results **findings**, of which there are two types:

- **Deviation**: two provisions that cover the same topic but don't have the same legal effect.
- **Gap**: a provision in one document with no agreement or difference in the other document.

For the purposes of this demo, provisions that agree with each other are ignored; this means we're only flagging items that may provide a source of conflict.

The general process is as follows:

1. Documents are prepared beforehand, i.e. cleaned and then normalized.
2. The documents are split into segments with [Kanon 2 Enricher](https://isaacus.com/docs/models/introduction#enrichment).
3. The segments are [embedded](https://isaacus.com/blog/what-are-vector-embeddings) with [Kanon 2 Embedder](https://isaacus.com/docs/models/introduction#embedding).
4. The segments get filtered with [Kanon Universal Classifier](https://isaacus.com/docs/models/introduction#universal-classification) to leave only relevant provisions (dropping things like footnotes).
5. Every provision in an input document gets matched against every provision in a reference document, and the closest pairs are shortlisted.
6. Each shortlisted pair is scored for conflict with the universal classifier and [Kanon 2 Reranker](https://isaacus.com/docs/models/introduction#reranking).
7. The pairs are then sorted into deviations and gaps.

_The pipeline diagram is in [the post](https://isaacus.com/blog/build-a-legal-document-conflict-detector)._

Our app will be a bit more involved. The code is split into three parts:
- Document indexing
- Document analysis
- Generation (optional)

We'll focus more on the first two as they cover finding deviations and gaps.

_Document indexing_ runs once per document:

1. **Normalize** the documents. Documents are converted into markdown and cleaned (encoding, stray characters, and whitespace). It's basic pre-processing.
2. **Enrich** the text into provisions (clauses).
3. **Embed** every clause.

We also **cache** the results to test values for future steps instead of making more API calls.

_Document analysis_ runs once per pair of documents (an input document and a reference document):

1. **Read** both documents through _document indexing_.
2. **Filter** out what isn't a provision, such as footnotes, citations and signature blocks, using the enricher's labels and the classifier. Definition clauses skip the classifier.
3. **Match** every input clause against every reference clause and shortlist the closest.
4. **Score** each shortlisted pair for conflict with the classifier and the reranker.
5. **Compare** provision pairs and decide which are deviations and which clauses are gaps, in document order. Definition clauses are ignored.
6. **Assemble** the report, then run the entire backend. A pair that hasn't changed since its last run reuses the saved report.

The third part, _generation_, is optional. It writes a summary of each finding using a gen AI model: a sentence or two on the differences between the two provisions. It also writes a recommendation for each finding about whether to keep, remove, or rewrite it. I briefly cover how it works, but we'll just be dropping in the whole file to see how it works with everything else.

## Prerequisites

<BlogCodeLanguagePicker />

- Node.js 24+ and npm.
- An [Isaacus account](https://isaacus.com/docs/quickstart#1-set-up-your-account) and [API key](https://isaacus.com/docs/quickstart#1-set-up-your-account).
- Sample documents. I've prepared a set in the repo's [`samples`](https://github.com/isaacus-dev/cookbooks/tree/main/cookbooks/conflict-detector/source_files/samples) folder, already converted to markdown. Your own documents can be plain text: the loader converts them.
  - Any of them can be a **reference document** (the document to check against) or an **input document** (the document to check).

This tutorial compares two different agreements: the Common Paper Cloud Service Agreement as the reference document, and Mistral AI's Terms of Use as the input document.

## Setup

1. Clone the cookbooks repo, go to the conflict detector in your language, and install its dependencies.

```bash
git clone --depth 1 https://github.com/isaacus-dev/cookbooks.git
cd cookbooks/cookbooks/conflict-detector/source_files/js
npm install
```

2. Create a `.env` file with your API key in that folder.

```bash
ISAACUS_API_KEY=your_isaacus_api_key_here
```

## A. Index a document

Before we can compare documents for conflicts, we first have to identify the provisions in them. Every document goes through the same stages: load it, enrich it into clauses, embed those, and then cache everything.

### Stage 1: clean the documents

Legal documents can be messy with all sorts of different patterns and layouts that need particular attention, such as tables. The sample files for this demo were fairly straightforward so general 'cleaning' fixes the rest: broken encoding, curly quotes, non-breaking spaces, stray control characters, and trailing whitespace. Cleaning them up helps us get better results from the models and makes them look prettier (💅) when we present them.

Isaacus' models were trained on markdown, so typical layout markers, such as headings and text formatting, are digested appropriately. Our sample files have already been converted to markdown. Any other text file goes through [`text2markdown`](https://github.com/isaacus-dev/text2markdown), our library that turns plain text into markdown.

We'll focus on the _normalize_ step first. It takes a document's text and returns it as clean markdown.

This section uses the following dependencies:

- `ftfy` and `text2markdown` have no JavaScript packages, so our ports of them are in `helpers/`. `unicode.js` comes with them; JavaScript counts a character outside the Basic Multilingual Plane as two, Python and the API count it as one, so it keeps every offset in step.
- [`isaacus`](https://docs.isaacus.com/): the official SDK. `text2markdown.js` uses it to call the enricher.

![The Odyssey Batman meme](https://isaacus.com/blog/build-a-legal-document-conflict-detector/the-odyssey-batman.gif)

_We don't want something popping up as something it isn't._

1. Add the cleaner and the converter.

- There's no `ftfy` for JavaScript, so `ftfy.js` is a port of the fixes it makes: broken encoding, HTML entities, curly quotes, ligatures and full-width characters. On the samples, both produce the same text, character for character.

- The same goes for `text2markdown`, which is Python-only: `text2markdown.js` is a port of it, and on the samples (converted to plain text) it writes the same markdown as the Python library. Conversion is an API call, so `normalizeText()` is `async` and everything that calls it awaits it.

In [ ]:
import { fixText } from "./source_files/js/helpers/ftfy.js";
import { stripTextEnd } from "./source_files/js/helpers/unicode.js";
import { text2markdown } from "./source_files/js/helpers/text2markdown.js";

const NON_STANDARD_CONTROL_CHARS_RE = /[\x07\x08\x0c\r\x0b]/g;
const START_OF_TEXT_THAT_IS_ONLY_WHITESPACE_FOLLOWED_BY_NEWLINE_RE = /^\s*\n/;
const END_OF_TEXT_THAT_IS_ONLY_WHITESPACE_PRECEDED_BY_NEWLINE_RE = /\n\s*$/;

function cleanText(text) {
  text = fixText(text);
  text = text.replaceAll("\xa0", " ");
  text = text.replace(NON_STANDARD_CONTROL_CHARS_RE, "");
  text = text.split("\n").map((line) => stripTextEnd(line)).join("\n");
  text = text.replace(START_OF_TEXT_THAT_IS_ONLY_WHITESPACE_FOLLOWED_BY_NEWLINE_RE, "");

  return text.replace(END_OF_TEXT_THAT_IS_ONLY_WHITESPACE_PRECEDED_BY_NEWLINE_RE, "");
}

// A document can be loaded more than once in a run; convert it only once
const markdownCache = new Map();

async function toMarkdown(text) {
  if (!markdownCache.has(text)) {
    // Skip cross-reference links: they don't help the models, and their HTML would stop ftfy decoding entities
    markdownCache.set(text, await text2markdown(text, { linkXrefs: false }));
  }

  return markdownCache.get(text);
}

async function normalizeText(text, { markdown }) {
  if (!markdown) {
    text = await toMarkdown(text);
  }

  return cleanText(text);
}

2. Test the cleaner by printing the first few lines:

In [ ]:
import fs from "node:fs";

const docPath = "source_files/samples/03-common-paper-cloud-service-agreement.md";
const raw = fs.readFileSync(docPath, "utf8");
const text = await normalizeText(raw, { markdown: true });

for (const line of text.split("\n").filter(Boolean).slice(0, 6)) {
  console.log(`  ${line.slice(0, 100)}`);
}

Expected output:

```text
  # Cloud Service Agreement
  ## 1. Service
  ### 1.1 Access and Use.
  During the Subscription Period and subject to the terms of this Agreement, Customer may (a) access a
  ### 1.2 Support.
  During the Subscription Period, Provider will provide Technical Support as described in the Order Fo
```

- Curly quotes are straightened, and trailing whitespace is gone.
- The markdown is still there: the `#` headings and the section numbers are untouched.

### Stage 2: enrich into clauses

We compare documents provision by provision, so first we need to find where each provision starts and ends. Kanon 2 Enricher turns a wall of text into structured segments: headings, containers, clauses, definition clauses, and list items. Segments can nest, so this stage removes nested text from each one and gives it a heading.

This section uses one dependency: [`dotenv`](https://www.npmjs.com/package/dotenv), which loads `ISAACUS_API_KEY` out of `.env`, so the key never lands in your code.

1. Start _document indexing_ with the imports, the paths, the clause record and the loader.

   - `text` is what we embed and classify.
   - `start` and `end` are character offsets in the document's text, so a report can point at the exact words rather than printing them again.
   - `type` and `category` are the enricher's labels for the segment. `category` is how we distinguish body text from front matter, back matter and annotations.
   - `loadText()` reads a document from disk. It hands the text to `normalizeText()`.

In [ ]:
import path from "node:path";

import dotenv from "dotenv";

const PROJECT_ROOT = path.join(process.cwd(), "source_files", "js");
const REPO_ROOT = path.dirname(PROJECT_ROOT);
const SAMPLES_DIR = path.join(REPO_ROOT, "samples");

const MARKDOWN_SUFFIXES = [".md", ".markdown"];

const ENRICHER_MODEL = "kanon-2-enricher";

void dotenv.config({ path: path.join(PROJECT_ROOT, ".env"), quiet: true });

function createClause({ doc, seg_id: segId, type, text, start = null, end = null, parent = null,
  category = null, heading = null, order = null, definition = false }) {
  return { doc, seg_id: segId, type, text, start, end, parent, category, heading, order, definition };
}

async function loadText(docPath) {
  const text = fs.readFileSync(docPath, "utf8").replace(/\r\n?/g, "\n");

  return normalizeText(text, { markdown: MARKDOWN_SUFFIXES.includes(path.extname(docPath).toLowerCase()) });
}

2. Create the API client. Every function that calls a model takes it as its first argument, so importing a module never connects to anything on its own.

In [ ]:
import Isaacus from "isaacus";

const client = new Isaacus({ apiKey: process.env.ISAACUS_API_KEY });

3. Add the enricher call, the section list and the heading resolver.

   - `resolveHeadings()` gives each clause the title of its nearest titled ancestor, or its own. Where the enricher finds no title at all, the fallback is the nearest markdown heading above the clause, without its number or bold markers. Headings only label findings. In the interactive demo, they label the cards.

In [ ]:
import { sliceText, stripText, textLength } from "./source_files/js/helpers/unicode.js";

async function enrichDocument(client, text) {
  return (await client.enrichments.create({
    model: ENRICHER_MODEL,
    texts: text,
    overflow_strategy: "auto",
  })).results[0].document;
}

function documentSections(document) {
  // The document's titled containers, saved with the index
  const sections = [];

  for (const seg of document.segments) {
    if (seg.kind !== "container" || !seg.title) {
      continue;
    }

    const title = stripText(sliceText(document.text, seg.title.start, seg.title.end));

    if (!title) {
      continue;
    }

    sections.push({
      title, level: seg.level,
      start: seg.span.start, end: seg.span.end,
    });
  }

  return sections;
}

function ownTitle(document, seg) {
  return seg.title ? stripText(sliceText(document.text, seg.title.start, seg.title.end)) : null;
}

const MARKDOWN_HEADING = /^#+[\t-\r\x1c-\x20\x85\p{Z}]+(.+)$/gmu;
const HEADING_NUMBER = /^\p{Nd}+(?:\\?\.\p{Nd}+)*\\?\.?[\t-\r\x1c-\x20\x85\p{Z}]+/u;
const DEFINITION_LEAD = /^(?:[^\p{L}\p{N}]|[\p{Nd}_])*(?:(?:the|an?)(?![\p{L}\p{N}_])[^\p{L}\p{N}]*)?/iu;

function markdownHeadings(text) {
  const found = [];

  for (const match of text.matchAll(MARKDOWN_HEADING)) {
    const title = stripText(match[1].replaceAll("*", "").replace(HEADING_NUMBER, ""));

    if (title) {
      found.push([textLength(text.slice(0, match.index)), title]);
    }
  }

  return found;
}

function resolveHeadings(document) {
  const markdown = markdownHeadings(document.text);
  const segments = new Map(document.segments.map((seg) => [seg.id, seg]));
  const headings = {};

  for (const seg of document.segments) {
    let [parentId, heading] = [seg.parent, null];

    while (parentId !== null && segments.has(parentId)) {
      heading = ownTitle(document, segments.get(parentId));

      if (heading) {
        break;
      }

      parentId = segments.get(parentId).parent;
    }

    // Title of the closest parent section that has one, else its own title, else the nearest markdown heading above
    headings[seg.id] = heading || ownTitle(document, seg)
      || (markdown.findLast(([start]) => start <= seg.span.start)?.[1] ?? null);
  }

  return headings;
}

4. Add the clause extractor.

   - The enricher returns segments nested inside each other: a section, the clauses beneath it, and the list items beneath those. Each segment's span covers its children's text too, so the same words would be counted more than once. `extractClauses()` cuts the children's spans out, so each segment keeps only its own text.
   - The enricher also marks each segment's own label: its type name ("Section"), its code ("4.2") and its title. `hasBody()` drops a piece with nothing left outside those, such as a bare heading.
   - The enricher doesn't mark every heading's title or number, so a markdown heading line (one starting with `#`) counts as a label too. Without it, "## Definitions" would be a clause of its own, which we don't want.
   - `definesTerm()` marks a clause as a definition when it opens with a term the enricher found, such as in `"Fees" means…`.

In [ ]:
import { stripTextStart } from "./source_files/js/helpers/unicode.js";

function opening(text) {
  return text.slice(DEFINITION_LEAD.exec(text)[0].length);
}

function definesTerm(document, { start, end }) {
  const clauseOpening = opening(sliceText(document.text, start, end));

  // A definition opens with a term the enricher found, and that term is defined inside this clause
  return document.terms.some((term) => (
    clauseOpening.startsWith(opening(sliceText(document.text, term.name.start, term.name.end)))
    && [term.name.start, term.meaning.start].some((pos) => start <= pos && pos < end)
  ));
}

function hasBody(document, seg, { start, end }) {
  // Spans that only label the segment: its type name ("Section"), code ("4.2") and title
  const labels = [seg.type_name, seg.code, seg.title].filter(Boolean).map((span) => [span.start, span.end]);

  // Markdown heading lines count as labels too
  let pos = start;

  for (const line of sliceText(document.text, start, end).split("\n")) {
    if (stripTextStart(line).startsWith("#")) {
      labels.push([pos, pos + textLength(line)]);
    }

    pos += textLength(line) + 1;
  }

  // It has a body if any letter or digit sits outside every label
  for (const [offset, char] of Array.from(sliceText(document.text, start, end)).entries()) {
    const n = start + offset;

    if (/[\p{L}\p{N}]/u.test(char) && !labels.some(([a, b]) => a <= n && n < b)) {
      return true;
    }
  }

  return false;
}

function extractClauses(docName, document) {
  const headings = resolveHeadings(document);
  const spans = new Map(document.segments.map((seg) => [seg.id, seg.span]));
  const clauses = [];

  for (const seg of document.segments) {
    const pieces = [];
    let cut = seg.span.start;

    // Cut the segment around its children's spans; each word belongs to one clause
    for (const child of seg.children.map((c) => spans.get(c)).sort((x, y) => x.start - y.start)) {
      pieces.push([cut, child.start]);
      cut = Math.max(cut, child.end);
    }

    pieces.push([cut, seg.span.end]);

    for (const [n, [start, end]] of pieces.entries()) {
      const raw = sliceText(document.text, start, end);
      const clauseText = stripText(raw);

      if (!hasBody(document, seg, { start, end })) {
        continue;
      }

      // Offsets of the text once leading and trailing whitespace is trimmed
      const clauseStart = start + textLength(raw) - textLength(stripTextStart(raw));
      const clauseEnd = start + textLength(stripTextEnd(raw));
      clauses.push(createClause({
        doc: docName, seg_id: n === 0 ? seg.id : `${seg.id}.${n}`, type: seg.type,
        text: clauseText, start: clauseStart, end: clauseEnd, parent: seg.parent,
        category: seg.category, heading: headings[seg.id],
        definition: definesTerm(document, { start: clauseStart, end: clauseEnd }),
      }));
    }
  }

  clauses.sort((x, y) => x.start - y.start);

  for (const [order, clause] of clauses.entries()) {
    clause.order = order;
  }

  return clauses;
}

5. This is the first call that spends credits, so it runs on one document. Test the enricher by printing the first few clauses:

In [ ]:
const docPath = path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md");
const document = await enrichDocument(client, await loadText(docPath));
const clauses = extractClauses(path.basename(docPath), document);

console.log(`${document.segments.length} segments -> ${clauses.length} clauses`);

for (const clause of clauses.slice(0, 5)) {
  console.log(`  ${clause.seg_id.padEnd(8)} [${clause.heading}] ${clause.text.slice(0, 80)}`);
}

Expected output:

```text
177 segments -> 106 clauses
  seg:2    [Access and Use] During the Subscription Period and subject to the terms of this Agreement, Custo
  seg:4    [Support] During the Subscription Period, Provider will provide Technical Support as descr
  seg:6    [User Accounts] Customer is responsible for all actions on Users' accounts and for all Users' co
  seg:8    [Feedback and Usage Data] Customer may, but is not required to, give Provider Feedback, in which case Cust
  seg:10   [Customer Content] Provider may copy, display, modify, and use Customer Content only as needed to p
```

- There should be fewer clauses than segments: sections that only hold other clauses, and bare headings, are dropped. If the numbers match, nothing was dropped.
- Each printed clause should show its heading in brackets. If every heading is `null`, `resolveHeadings()` isn't finding them.

### Stage 3: embed and cache the clauses

Time to convert text into embeddings to be sorted, searched and compared. We're using a semantic embedding model, which lets us measure how close two clauses are in meaning and not just wording.

A `Float32Array` holds the embeddings as one matrix, compares them with a dot product, and is saved to disk as an `.npy` file.

1. Add the model, the batch size and the normalizer.

   - The embedder turns each clause into a vector. Clauses about the same thing end up 'close' to each other, and cosine similarity scores that closeness.
   - `unitNormalize()` scales every vector to length 1. Cosine similarity is the **dot product** (multiply matching numbers and add them up) divided by both vectors' lengths, so when every length is 1, it's just the dot product. Because the vectors are scaled before they're saved, comparing every clause in one document with every clause in another takes a single matrix multiplication.

In [ ]:
const EMBEDDER_MODEL = "kanon-2-embedder";
const EMBED_BATCH_SIZE = 96;

function unitNormalize(rows) {
  const vectors = { data: Float32Array.from(rows.flat()), shape: rows.length ? [rows.length, rows[0].length] : [0, 0] };

  if (vectors.data.length === 0) {
    return vectors;
  }

  const [count, dims] = vectors.shape;

  for (let r = 0; r < count; r++) {
    const row = vectors.data.subarray(r * dims, (r + 1) * dims);

    // Divide each vector by its length; an all-zero vector stays zero rather than dividing by zero
    const norm = Math.sqrt(row.reduce((sum, x) => sum + x * x, 0)) || 1;

    for (let c = 0; c < dims; c++) {
      row[c] /= norm;
    }
  }

  return vectors;
}

async function embedPropositions(client, texts, { task = "retrieval/document" } = {}) {
  const rows = [];

  for (let i = 0; i < texts.length; i += EMBED_BATCH_SIZE) {
    const resp = await client.embeddings.create({
      model: EMBEDDER_MODEL,
      texts: texts.slice(i, i + EMBED_BATCH_SIZE),
      task,
    });
    rows.push(...resp.embeddings.map((e) => e.embedding));
  }

  return unitNormalize(rows);
}

2. Test the embeddings by printing the clauses closest to the payment clause:

In [ ]:
const docPath = path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md");
const document = await enrichDocument(client, await loadText(docPath));
const clauses = extractClauses(path.basename(docPath), document);
const vectors = await embedPropositions(client, clauses.map((c) => c.text));
const [count, dims] = vectors.shape;

console.log(`${clauses.length} clauses -> ${count} vectors`);

const n = clauses.findIndex((clause) => clause.heading === "Payment");
const query = vectors.data.subarray(n * dims, (n + 1) * dims);
const scores = clauses.map((_, m) => query.reduce((sum, x, d) => sum + x * vectors.data[m * dims + d], 0));
console.log(`closest to [${clauses[n].heading}] ${clauses[n].text.slice(0, 60)}`);

const closest = scores.map((score, m) => [score, m]).sort((a, b) => b[0] - a[0]).slice(0, 4);

for (const [score, m] of closest) {
  console.log(`  ${score.toFixed(2)}  [${clauses[m].heading}] ${clauses[m].text.slice(0, 60)}`);
}

Expected output:

```text
106 clauses -> 106 vectors
closest to [Payment] Customer will pay Provider Fees and taxes in U.S. Dollars, u
  1.00  [Payment] Customer will pay Provider Fees and taxes in U.S. Dollars, u
  0.67  [Fees] Unless the Order Form specifies a different currency, all Fe
  0.61  [Taxes] Customer is responsible for all duties, taxes, and levies th
  0.58  [Invoicing] For a Payment Process with invoicing, Provider will send inv
```

- Every clause should get one vector, so the two counts match.
- The closest clauses should be about the same thing: here, fees, taxes and invoicing.
- The clause itself comes first with a score of `1.00`. Anything else and `unitNormalize()` isn't being applied.

#### Cache the index

Unlike Penelope's loom situation, we don't want to have to repeat anything more than necessary.

![Penelope at her Loom, painting by Angelica Kauffman](https://isaacus.com/blog/build-a-legal-document-conflict-detector/Angelica_Kaufmann_-_Penelope_at_her_Loom.jpg)

_Some of us probably bear the same expression unpicking vibe code instead of a loom._

Enriching and embedding a document only depends on its text, so the result is saved under a hash of that text, not under the file's name or path. Rename or move a document and it's still found in the cache; only a change to its text sends it back to the models.

1. Add the cache: the index record, a hash function, a reader, a writer, the document records, and the function that ties the stages together.

   - Bump `PIPELINE_VERSION` by hand when you change how clauses are extracted. The text hasn't changed, so nothing else would.
   - An index is five files: `embeddings.npy`, `propositions.json`, the cleaned text, the sections and a manifest. Row `j` of the embeddings belongs to proposition `j`, so `readIndex()` refuses the pair if their counts differ.

In [ ]:
import crypto from "node:crypto";

const RESULTS_DIR = path.join(PROJECT_ROOT, "results");
const DOC_CACHE_DIR = path.join(RESULTS_DIR, "cache");
const DOCUMENTS_DIR = path.join(RESULTS_DIR, "documents");

const DOCUMENT_SCHEMA = "conflict-document/1";

const PIPELINE_VERSION = 19;

function createDocumentIndex({ propositions, embeddings, manifest = null, text = null, sections = null }) {
  // One document's clauses (called propositions here), their embeddings and its cleaned text
  return { propositions, embeddings, manifest, text, sections };
}

function npLoad(file) {
  const buf = fs.readFileSync(file);
  const end = 10 + buf.readUInt16LE(8);
  const shape = /'shape': \(([^)]*)\)/.exec(buf.toString("latin1", 10, end))[1];

  return {
    data: new Float32Array(buf.buffer.slice(buf.byteOffset + end, buf.byteOffset + buf.length)),
    shape: shape.split(",").filter((n) => n.trim()).map(Number),
  };
}

function npSave(file, { data, shape }) {
  const dict = `{'descr': '<f4', 'fortran_order': False, 'shape': (${shape.join(", ")}), }`;
  const header = `${dict.padEnd(dict.length + 63 - ((10 + dict.length) % 64))}\n`;
  const size = Buffer.alloc(2);
  size.writeUInt16LE(header.length);
  fs.writeFileSync(file, Buffer.concat([Buffer.from("\x93NUMPY\x01\x00", "latin1"), size,
    Buffer.from(header, "latin1"), Buffer.from(data.buffer, data.byteOffset, data.byteLength)]));
}

function readIndex(indexDir) {
  let rows;
  let matrix;
  let manifest;
  let text;
  let sections;

  try {
    rows = JSON.parse(fs.readFileSync(path.join(indexDir, "propositions.json"), "utf8"));
    matrix = npLoad(path.join(indexDir, "embeddings.npy"));
    manifest = JSON.parse(fs.readFileSync(path.join(indexDir, "manifest.json"), "utf8"));
    text = fs.readFileSync(path.join(indexDir, "text.txt"), "utf8");
    sections = JSON.parse(fs.readFileSync(path.join(indexDir, "sections.json"), "utf8"));
  } catch {
    return null;
  }

  if (matrix.shape.length !== 2 || matrix.shape[0] !== rows.length) {
    return null;
  }

  return createDocumentIndex({
    propositions: rows.map((row) => createClause(row)),
    embeddings: matrix,
    manifest,
    text,
    sections,
  });
}

function writeIndex(index, indexDir) {
  fs.mkdirSync(indexDir, { recursive: true });
  fs.writeFileSync(
    path.join(indexDir, "manifest.json"), JSON.stringify(index.manifest, null, 2), "utf8",
  );
  fs.writeFileSync(
    path.join(indexDir, "propositions.json"),
    JSON.stringify(index.propositions, null, 2),
    "utf8",
  );
  npSave(path.join(indexDir, "embeddings.npy"), index.embeddings);
  fs.writeFileSync(path.join(indexDir, "text.txt"), index.text, "utf8");
  fs.writeFileSync(
    path.join(indexDir, "sections.json"), JSON.stringify(index.sections || [], null, 2), "utf8",
  );

  return indexDir;
}

async function documentIdentity(docPath, index, { text = null } = {}) {
  if (text === null) {
    text = index.text !== null ? index.text : await loadText(docPath);
  }

  const rel = path.relative(fs.realpathSync(REPO_ROOT), fs.realpathSync(docPath)).split(path.sep).join("/");

  return {
    source: rel === ".." || rel.startsWith("../") || path.isAbsolute(rel) ? String(docPath) : rel,
    source_sha256: crypto.createHash("sha256").update(fs.readFileSync(docPath)).digest("hex"),
    text_sha256: crypto.createHash("sha256").update(text, "utf8").digest("hex"),
    sections: index.sections || [],
  };
}

async function documentArtifact(docPath, index) {
  let text = await loadText(docPath);

  if (!(index.text === null || index.text === text)) {
    throw new Error(`${path.basename(docPath)}: clear results/cache`);
  }

  text = index.text !== null ? index.text : text;

  return { schema: DOCUMENT_SCHEMA, ...(await documentIdentity(docPath, index, { text })), text };
}

async function writeDocumentArtifact(docPath, index, { documentsDir = DOCUMENTS_DIR } = {}) {
  // The report only stores offsets, so the demo reads each document's text from here
  const payload = await documentArtifact(docPath, index);
  const dest = path.join(documentsDir, `${payload.source_sha256}.json`);
  fs.mkdirSync(path.dirname(dest), { recursive: true });
  fs.writeFileSync(dest, JSON.stringify(payload, null, 2) + "\n", "utf8");

  return dest;
}

function documentFingerprint(normalizedText) {
  // Any change to the text, the pipeline version or a model re-indexes; delete results/cache/ to force it
  const stamp = `${PIPELINE_VERSION}|${ENRICHER_MODEL}|${EMBEDDER_MODEL}|${normalizedText}`;

  return crypto.createHash("sha256").update(stamp, "utf8").digest("hex").slice(0, 16);
}

function documentManifest(docName, docPath, entry) {
  return { doc_name: docName, source_path: String(docPath), files: { [docName]: entry } };
}

async function indexDocument(client, docPath, { useCache = true } = {}) {
  const text = await loadText(docPath);
  const cacheDir = path.join(DOC_CACHE_DIR, documentFingerprint(text));

  if (useCache) {
    const cached = rekey(readIndex(cacheDir), path.basename(docPath), docPath);

    if (cached !== null) {
      return cached;
    }
  }

  const document = await enrichDocument(client, text);
  const propositions = extractClauses(path.basename(docPath), document);

  const index = createDocumentIndex({
    propositions,
    embeddings: await embedPropositions(client, propositions.map((p) => p.text)),
    text: document.text,
    sections: documentSections(document),
    manifest: documentManifest(path.basename(docPath), docPath, {
      proposition_count: propositions.length, clause_count: propositions.length }),
  });
  writeIndex(index, cacheDir);

  return index;
}

function rekey(index, docName, docPath) {
  if (index === null || index.text === null) {
    return null;
  }

  // A cached index may have been saved under another file name
  for (const clause of index.propositions) {
    clause.doc = docName;
  }

  const entry = Object.values((index.manifest || {}).files ?? {})[0] ?? {};
  index.manifest = documentManifest(docName, docPath, entry);

  return index;
}

2. The earlier tests called the enricher directly, so they spent credits on every run; this goes through the cache. Test the cache by indexing the same document twice and timing each run:

In [ ]:
import { performance } from "node:perf_hooks";

const docPath = path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md");

for (const attempt of ["cold", "warm"]) {
  const started = performance.now();
  const index = await indexDocument(client, docPath);
  const seconds = ((performance.now() - started) / 1000).toFixed(2);

  console.log(`${attempt.padEnd(5)} ${seconds.padStart(6)}s  ${index.propositions.length} propositions`);
}

Expected output:

```text
cold    3.42s  106 propositions
warm    0.00s  106 propositions
```

- The warm run should take a fraction of a second and report the same count. If both runs take the same time, the cache is never being read: check `results/cache/` has a folder in it.

3. Index the input document. It needs to run through the same process as the reference in order to be compared to it. Test the index by printing how many propositions each document has:

In [ ]:
const reference = await indexDocument(client, path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md"));
const input = await indexDocument(client, path.join(SAMPLES_DIR, "04-mistral-ai-terms-of-use.md"));

for (const [name, index] of [["reference", reference], ["input", input]]) {
  console.log(`${name.padEnd(10)} ${String(index.propositions.length).padStart(4)} propositions`);
}

Expected output:

```text
reference   106 propositions
input       251 propositions
```

Neither count should be zero.

## B. Analyze documents

Everything below is part of _document analysis_. We're essentially going to build a search engine, but for conflicts. We've already embedded our provisions, so this is where we compare them and classify conflicts.

Penelope gets suitors; we'd probably just get a suit. At least we can also get rid of it in minutes instead of two decades – much like how it feels to work on some documents (or blog posts 👀).

### Stage 1: filter and match

This is the bulk of the conflict search engine. Scoring every clause against every other clause would take thousands of model calls, so we narrow them down first.

Footnotes, citations and signature blocks aren't provisions, so they're dropped. The rest are matched by embedding similarity, with no extra model calls. The closest pairs go on to the classifier and reranker in batches. Think of it as cutting Penelope's suitors before the feast.

![A Trojan horse washed up on a beach](https://isaacus.com/blog/build-a-legal-document-conflict-detector/The-Odyssey-Trojan-Horse-Beach.webp)

_Don't want anything sneaking in, do we?_

1. Add the classifier call, the filter, the matrix and the shortlist.

   - The filter has two parts. The enricher already labels front matter, back matter and annotations, so those are filtered immediately. The classifier scores every other clause against one [IQL](https://docs.isaacus.com/iql/introduction) statement, roughly "this is a footnote, citation or signature, not a provision", and sets aside anything at or above `FILTER_THRESHOLD`.
   - Definitions are never filtered. The classifier scores them inconsistently, and a changed definition can matter as much as any obligation. Stage 2 weighs them down instead.
   - `shortlistPairs()` keeps a pair if it scores at least `SHORTLIST_MIN_RELEVANCE` and is one of the top `SHORTLIST_TOP_K` matches for either clause. Ranking from both sides means every reference clause gets its own best matches, so one missing from the input can still be reported.

In [ ]:
import { splitText } from "./source_files/js/helpers/unicode.js";

const CLASSIFIER_MODEL = "kanon-universal-classifier";

const SHORTLIST_MIN_RELEVANCE = 0.5;
const SHORTLIST_TOP_K = 3;

const STRUCTURAL_CATEGORIES = ["front_matter", "back_matter", "annotation"];

const FILTER_QUERY = "{This is a footnote, citation, editorial note, heading or signature, "
  + "not a contractual provision.}";
const FILTER_THRESHOLD = 0.5;

function pairMatrix(input, reference) {
  if (input.embeddings.data.length === 0 || reference.embeddings.data.length === 0) {
    return input.propositions.map(() => new Array(reference.propositions.length).fill(0));
  }

  const [a, b, dims] = [input.embeddings.data, reference.embeddings.data, input.embeddings.shape[1]];

  return input.propositions.map((_, i) => reference.propositions.map((_, j) => {
    let dot = 0;

    for (let k = 0; k < dims; k++) {
      dot += a[i * dims + k] * b[j * dims + k];
    }

    return Math.fround(dot);
  }));
}

async function nonOperative(client, index) {
  const scores = await classify(client, FILTER_QUERY, index.propositions.map((c) => flatten(c.text)), { isIql: true });

  return index.propositions.map((c, n) => (
    STRUCTURAL_CATEGORIES.includes(c.category) || (!c.definition && scores[n] >= FILTER_THRESHOLD)
  ));
}

function maskNonOperative(sim, inc, ref) {
  // A filtered clause scores -1 with everything, so it can't be shortlisted
  return sim.map((row, i) => row.map((value, j) => (inc[i] || ref[j] ? -1.0 : value)));
}

function shortlistPairs(sim, { minRelevance, topK }) {
  if (sim.flat().length === 0) {
    return new Map();
  }

  function rank(values) {
    const ranks = new Array(values.length);
    values.map((_, n) => n).sort((x, y) => values[y] - values[x] || x - y).forEach((n, r) => {
      ranks[n] = r;
    });

    return ranks;
  }

  // Rank of each pair among its input clause's matches, and among its reference clause's (0 = closest)
  const rowRank = sim.map(rank);
  const colRank = sim[0].map((_, j) => rank(sim.map((row) => row[j])));

  const shortlist = new Map();
  sim.forEach((row, i) => row.forEach((value, j) => {
    // Keep a pair that's similar enough and in the top k for either clause
    if (value >= minRelevance && (rowRank[i][j] < topK || colRank[j][i] < topK)) {
      if (!shortlist.has(i)) {
        shortlist.set(i, []);
      }

      shortlist.get(i).push([j, value]);
    }
  }));

  for (const row of shortlist.values()) {
    row.sort((x, y) => y[1] - x[1]);
  }

  return shortlist;
}

function flatten(text) {
  return splitText(text).join(" ");
}

function span(clause) {
  if (clause.start === null || clause.end === null) {
    return null;
  }

  return { start: clause.start, end: clause.end };
}

function clauseRef(clause) {
  return { seg_id: clause.seg_id, text: flatten(clause.text), span: span(clause) };
}

const CLASSIFY_MAX_PARALLEL_REQUESTS = 6;
const CLASSIFY_BATCH_SIZE = 50;

async function mapPool(items, concurrency, worker) {
  // Runs worker over items, at most concurrency at a time: the stand-in for Python's ThreadPoolExecutor
  const results = new Array(items.length);
  let cursor = 0;

  async function runner() {
    while (cursor < items.length) {
      const i = cursor++;
      results[i] = await worker(items[i], i);
    }
  }

  await Promise.all(Array.from({ length: Math.min(concurrency, items.length) }, runner));

  return results;
}

async function batched(score, texts) {
  const batches = [];

  for (let n = 0; n < texts.length; n += CLASSIFY_BATCH_SIZE) {
    batches.push(texts.slice(n, n + CLASSIFY_BATCH_SIZE));
  }

  return (await mapPool(batches, CLASSIFY_MAX_PARALLEL_REQUESTS, score)).flat();
}

async function classifyBatch(client, batch, { query, isIql }) {
  const resp = await client.classifications.universal.create({
    model: CLASSIFIER_MODEL, query, texts: batch, is_iql: isIql, scoring_method: "auto",
  });

  return [...resp.classifications].sort((a, b) => a.index - b.index).map((c) => c.score);
}

async function classify(client, query, texts, { isIql = false } = {}) {
  return batched((batch) => classifyBatch(client, batch, { query, isIql }), texts);
}

2. Both documents come out of the cache; the filter is the part that calls the classifier. Test the filter by counting how many clauses it drops and how many candidate pairs are left:

In [ ]:
const reference = await indexDocument(client, path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md"));
const input = await indexDocument(client, path.join(SAMPLES_DIR, "04-mistral-ai-terms-of-use.md"));
const inputFiltered = await nonOperative(client, input);
const referenceFiltered = await nonOperative(client, reference);
const sim = pairMatrix(input, reference);

const shortlist = shortlistPairs(maskNonOperative(sim, inputFiltered, referenceFiltered), {
  minRelevance: SHORTLIST_MIN_RELEVANCE, topK: SHORTLIST_TOP_K,
});

const pairs = [...shortlist.values()].reduce((n, row) => n + row.length, 0);

for (const [name, filtered] of [["input", inputFiltered], ["reference", referenceFiltered]]) {
  const count = String(filtered.filter(Boolean).length).padStart(3);
  console.log(`${name.padEnd(10)} ${count} of ${filtered.length} clauses filtered`);
}

console.log(`candidate pairs: ${pairs}`);

Expected output:

```text
input       24 of 251 clauses filtered
reference    1 of 106 clauses filtered
candidate pairs: 197
```

- About a tenth of the input is filtered: headings, cross-references and other text that isn't a provision. The reference is almost all provisions, so only one clause goes. If nearly everything is filtered, `FILTER_THRESHOLD` is too low.
- Candidate pairs are what go on to the classifier and reranker, about 200 here. If there are thousands, raise `SHORTLIST_MIN_RELEVANCE` or lower `SHORTLIST_TOP_K`.
- Your counts may differ by one or two between runs, because the embedder's output varies slightly.

### Stage 2: score

One of the few times it's good to be judgy: the universal classifier and the reranker earning their keep. In this stage we produce a score that quantifies how much two provisions conflict.

Every shortlisted pair gets two numbers:

- **$P(\text{same})$** is how likely the two clauses deal with the same matter. It's worked out from the embeddings we already have, so it needs no more model calls.
- **$P(\text{differs})$** is how likely the difference between them matters, from four questions put to the classifier and the reranker.
- Combined and calibrated into **$P(\text{conflict})$**, they decide what becomes a finding.

1. Add the questions, the weights and the scorer.

   - `matchMatrix()` turns the embedding similarity between two clauses into $P(\text{same})$. A similarity of 0.6 can be the best match in one pair of documents and an ordinary one in another, so the raw score isn't enough. Instead, a pair scores high only when each clause is clearly the other's closest match. A clause with no real counterpart has no clear closest match, so all its pairs score low.
   - Four questions decide whether a difference matters. `empty` (is it standard wording, a heading, a signature or a definition?) and `money` (does it state an amount, a percentage or a cap?) are asked of each clause. `care` and `agree` compare the two clauses directly.
   - `materiality` combines the four answers into one score. A clause `empty` flags, such as standard wording found in most contracts or a signature block, pulls it down hard; clauses that agree, or that a party wouldn't care about, pull it down a little; an amount pushes it up. `differs` turns that score into a probability between 0 and 1: $P(\text{differs})$.
   - `conflict` combines $P(\text{same})$ and $P(\text{differs})$ into $P(\text{conflict})$. Multiplying them is too cautious, so the combination is fitted: 0.73 means about a 73% chance of a real difference.
   - The `SAME_*`, `DIFFERS_*`, `CONFLICT_*` and `*_WEIGHT` constants are fitted on labeled clause pairs. With the questions, they're what you'd retune.
   - `CLASSIFY_QUERY_MAX_CHARS` trims the clause sent with `care`. Each distinct clause is sent once, however many pairs it's in.

In [ ]:
const RERANKER_MODEL = "kanon-2-reranker";

const EMPTY_QUERY = "routine boilerplate, a recital, a preamble, a heading, a signature block, "
  + "an address or a definition";
const CARE_STATEMENT = "A party would not care whether this clause or the following clause applied: {clause}";
const AGREE_QUERY = "A lawyer would compare this clause with the following clause: {clause}";
const MONEY_STATEMENT = "This states an amount of money, a percentage or a cap.";
const CLASSIFY_QUERY_MAX_CHARS = 2000;
const MATCH_TEMPERATURE = 0.06;
const SAME_SLOPE = 2.049;
const SAME_BIAS = 3.058;

const EMPTY_WEIGHT = -4.0;
const CARE_WEIGHT = -1.0;
const AGREE_WEIGHT = -1.0;
const MONEY_WEIGHT = 1.0;
const DIFFERS_SLOPE = 0.526;
const DIFFERS_BIAS = -12.312;
const CONFLICT_SAME_SLOPE = 0.576;

const CONFLICT_DIFFERS_SLOPE = 0.707;
const CONFLICT_BIAS = 1.663;

async function rerankBatch(client, batch, { query }) {
  const resp = await client.rerankings.create({ model: RERANKER_MODEL, query, texts: batch });

  return [...resp.results].sort((a, b) => a.index - b.index).map((r) => r.score);
}

async function rerank(client, query, texts) {
  return batched((batch) => rerankBatch(client, batch, { query }), texts);
}

function logit(p) {
  p = Math.min(Math.max(p, 1e-6), 1 - 1e-6);

  return Math.log(p / (1 - p));
}

function sigmoid(x) {
  return 1 / (1 + Math.exp(-x));
}

function logSigmoid(x) {
  // log(sigmoid(x)), without rounding tiny probabilities to zero
  return -(Math.max(0, -x) + Math.log1p(Math.exp(-Math.abs(x))));
}

function mean(scores, a, b) {
  return (scores.get(a) + scores.get(b)) / 2;
}

async function scoreQuery(b, { texts, score, template, maxChars }) {
  // Ask the model about clause b once, against every clause it's paired with
  const scores = await score(template.replace("{clause}", () => sliceText(b, 0, maxChars)), texts);

  return texts.map((a, n) => [JSON.stringify([a, b]), scores[n]]);
}

async function pairwise(score, template, pairs, { maxChars } = {}) {
  const byQuery = new Map();

  // Group pairs by their second clause, so each clause goes to the model once
  for (const [a, b] of new Map(pairs.map((pair) => [JSON.stringify(pair), pair])).values()) {
    if (!byQuery.has(b)) {
      byQuery.set(b, []);
    }

    byQuery.get(b).push(a);
  }

  const results = await mapPool([...byQuery], CLASSIFY_MAX_PARALLEL_REQUESTS,
    ([b, texts]) => scoreQuery(b, { texts, score, template, maxChars }));

  return new Map(results.flat());
}

function createPairScore({ relevance, match, empty, care, agree, money }) {
  return {
    relevance, match, empty, care, agree, money,
    get sameMatter() {
      return sigmoid(SAME_SLOPE * this.match + SAME_BIAS);
    },
    get materiality() {
      return EMPTY_WEIGHT * logit(this.empty) + CARE_WEIGHT * logit(this.care)
        + AGREE_WEIGHT * logit(this.agree) + MONEY_WEIGHT * logit(this.money);
    },
    get differs() {
      return sigmoid(DIFFERS_SLOPE * this.materiality + DIFFERS_BIAS);
    },
    get conflict() {
      return sigmoid(CONFLICT_SAME_SLOPE * logSigmoid(SAME_SLOPE * this.match + SAME_BIAS)
        + CONFLICT_DIFFERS_SLOPE * logSigmoid(DIFFERS_SLOPE * this.materiality + DIFFERS_BIAS)
        + CONFLICT_BIAS);
    },
  };
}

function matchMatrix(sim) {
  const logits = sim.map((row) => row.map((s) => s / MATCH_TEMPERATURE));

  function logSumExp(values) {
    const max = Math.max(...values);

    return max + Math.log(values.reduce((sum, v) => sum + Math.exp(v - max), 0));
  }

  const rows = logits.map(logSumExp);
  const cols = (logits[0] ?? []).map((_, j) => logSumExp(logits.map((row) => row[j])));

  return logits.map((row, i) => row.map((l, j) => ((l - rows[i]) + (l - cols[j])) / 2));
}

async function scorePairs(client, input, reference, { shortlist }) {
  const match = matchMatrix(pairMatrix(input, reference));
  const texts = new Map([...shortlist].flatMap(([i, row]) => row.map(([j]) => [
    `${i}:${j}`, [flatten(input.propositions[i].text), flatten(reference.propositions[j].text)],
  ])));
  const unique = [...new Set([...texts.values()].flat())];
  const bothWays = [...texts.values()].flatMap(([a, b]) => [[a, b], [b, a]]);
  const empty = new Map((await rerank(client, EMPTY_QUERY, unique)).map((score, n) => [unique[n], score]));
  const money = new Map((await classify(client, MONEY_STATEMENT, unique)).map((score, n) => [unique[n], score]));
  const care = await pairwise((query, texts) => classify(client, query, texts), CARE_STATEMENT, bothWays, {
    maxChars: CLASSIFY_QUERY_MAX_CHARS,
  });
  const agree = await pairwise((query, texts) => rerank(client, query, texts), AGREE_QUERY, bothWays);

  return new Map([...shortlist].flatMap(([i, candidates]) => candidates.map(([j, relevance]) => {
    const [a, b] = texts.get(`${i}:${j}`);

    return [`${i}:${j}`, createPairScore({
      relevance, match: match[i][j],
      empty: mean(empty, a, b), money: mean(money, a, b),
      care: mean(care, JSON.stringify([a, b]), JSON.stringify([b, a])),
      agree: mean(agree, JSON.stringify([a, b]), JSON.stringify([b, a])),
    })];
  })));
}

Scoring is the stage that costs money; you'll see its results in the Stage 4 run.

### Stage 3: compare

Stage 2 scored every shortlisted pair. This stage decides which of them are deviations and which clauses are gaps to complete our actual findings.

1. Add the thresholds, the row helpers, the partition and the two roll-ups.

   - `partition()` first drops any pair that isn't about the same topic (below `SAME_THRESHOLD`). Then every clause, on both sides, picks the one counterpart it matches best. If that pair's $P(\text{conflict})$ reaches `CONFLICT_THRESHOLD`, it's a deviation.
   - Only the best match counts. Otherwise a clause would get flagged against every vaguely similar clause nearby. No drowning in findings, thanks.
   - Picking from both sides matters because documents don't split things up the same way. If the reference spreads one input clause across four, all four pick it, so it ends up as one row with four matches.
   - By the end, every clause has one of three statuses:
     - `deviation`: its best match conflicts with it.
     - `covered`: it has a match, and they don't conflict.
     - `gap`: nothing in the other document covers the same thing.
   - `rollUpGaps()` turns gaps into findings for each side: what the input document dropped, and what it added. Filtered clauses never become gaps, and `covered` clauses are ignored; they agree, so there's nothing to flag.
   - A definition with nothing to match isn't a gap. Documents define different terms, or define them in different places: the input defines "Effective Date", while the reference just uses it because its cover page sets the date. If a term actually matters, the clauses that use it will turn up as findings anyway.

In [ ]:
const SAME_THRESHOLD = 0.3;

const CONFLICT_THRESHOLD = 0.728;

function status(pairs, { deviating }) {
  const best = pairs.reduce((top, e) => (top === null || e[1].relevance > top[1].relevance ? e : top), null);

  return {
    status: deviating ? "deviation" : pairs.length ? "covered" : "gap",
    best_relevance: best ? best[1].relevance : null,
    nearest_index: best ? best[0] : null,
  };
}

function sideStatus(index, pairs, deviating) {
  return Object.fromEntries(index.propositions.map((_, n) => [
    n, status(pairs.get(n) ?? [], { deviating: deviating.has(n) }),
  ]));
}

function partition(input, reference, scored) {
  const closest = [new Map(), new Map()];

  // For each clause on each side, keep its best match that's about the same matter
  for (const [key, sc] of scored) {
    const pair = key.split(":").map(Number);

    for (const [side, best] of closest.entries()) {
      if (sc.sameMatter >= SAME_THRESHOLD && (!best.has(pair[side]) || sc.match > best.get(pair[side])[1].match)) {
        best.set(pair[side], [key, sc]);
      }
    }
  }

  // Both sides' picks together; a pair both clauses picked appears once
  const selected = new Map(closest.flatMap((best) => [...best.values()]));

  // A picked pair is a deviation when its P(conflict) clears the threshold
  const deviatingPairs = [...selected].filter(([, sc]) => sc.conflict >= CONFLICT_THRESHOLD)
    .map(([key, sc]) => [...key.split(":").map(Number), sc]);

  const incPairs = new Map();
  const refPairs = new Map();

  for (const [key, score] of scored) {
    const [i, j] = key.split(":").map(Number);

    if (score.sameMatter >= SAME_THRESHOLD) {
      if (!incPairs.has(i)) {
        incPairs.set(i, []);
      }

      if (!refPairs.has(j)) {
        refPairs.set(j, []);
      }

      incPairs.get(i).push([j, score]);
      refPairs.get(j).push([i, score]);
    }
  }

  return [deviatingPairs,
    sideStatus(input, incPairs, new Set(deviatingPairs.map(([i]) => i))),
    sideStatus(reference, refPairs, new Set(deviatingPairs.map(([, j]) => j)))];
}

function clauseRow(clause) {
  return {
    id: null,
    seg_id: clause.seg_id,
    heading: clause.heading,
    text: flatten(clause.text),
    span: span(clause),
    order: clause.order,
    category: clause.category,
  };
}

function numbered(rows, sortKey) {
  // Number rows in document order, so the report reads top to bottom like the documents
  rows.sort(sortKey);
  rows.forEach((row, n) => {
    row.id = n + 1;
  });

  return rows;
}

function rollUpDeviations(deviatingPairs, input, reference) {
  const byInput = new Map();

  for (const [i, j, score] of deviatingPairs) {
    if (!byInput.has(i)) {
      byInput.set(i, []);
    }

    byInput.get(i).push([j, score]);
  }

  const rows = [];

  for (const [i, pairs] of byInput) {
    pairs.sort((a, b) => (reference.propositions[a[0]].order || 0) - (reference.propositions[b[0]].order || 0));
    rows.push({
      ...clauseRow(input.propositions[i]),
      relevance: pairs[0][1].relevance,
      match_count: pairs.length,
      matches: pairs.map(([j, score]) => ({
        input: clauseRef(input.propositions[i]),
        reference: {
          ...clauseRef(reference.propositions[j]),
          clause_text: flatten(reference.propositions[j].text),
        },
        relevance: score.relevance,
      })),
    });
  }

  return numbered(rows, (a, b) => (a.order || 0) - (b.order || 0));
}

function rollUpGaps(index, status, { other, filtered }) {
  const rows = [];

  for (const [i, clause] of index.propositions.entries()) {
    const info = status[i];

    if (filtered[i] || info.status !== "gap" || clause.definition) {
      continue;
    }

    const near = info.nearest_index;
    rows.push({
      ...clauseRow(clause),
      propositions: { total: 1, matched_count: 0, uncovered_count: 1 },
      gap_score: 1.0 - (info.best_relevance || 0.0),
      uncovered_propositions: [{
        ...clauseRef(clause),
        status: info.status,
        best_relevance: info.best_relevance,
        nearest: near !== null ? clauseRef(other.propositions[near]) : null,
      }],
    });
  }

  return numbered(rows, (a, b) => (a.order || 0) - (b.order || 0));
}

### Stage 4: report and run

The last stage puts every finding into one report and saves it under the pair it belongs to. Then we run the whole pipeline.

1. Add the report, the writer, and the check that reuses a saved run.

   - `report.json` holds the findings. `pairs.json` holds every shortlisted pair behind them, with all four answers and the three probabilities. Want to try a different threshold? You can see what would change without paying for the models again.
   - `params.json` is the paper trail: the models, the thresholds, the wording of every question, and the pipeline version. If two runs disagree, look here first.
   - `savedReportIfCurrent()` reuses a saved run as long as the settings and both documents' text haven't changed, so running the same pair twice doesn't hit the models again. The exception is a plain-text document: it still gets converted to markdown on every run, since that's how we get the text to compare.

In [ ]:
const REPORTS_DIR = path.join(RESULTS_DIR, "reports");

const SCHEMA = "doc-compare/2";

function coverageCounts(status, filtered) {
  const counts = {
    total: Object.keys(status).length, covered: 0, deviation: 0, gap: 0,
    filtered: filtered.filter(Boolean).length,
  };

  for (const [n, row] of Object.entries(status)) {
    if (!filtered[n]) {
      counts[row.status] += 1;
    }
  }

  return counts;
}

async function documentSummary(index, docPath) {
  return {
    name: path.basename(docPath),
    ...(await documentIdentity(docPath, index)),
    propositions: index.propositions.length,
  };
}

function scoredPairsRows(input, reference, scored, { deviating }) {
  const rows = [];

  for (const [key, score] of scored) {
    const [i, j] = key.split(":").map(Number);
    const [inc, ref] = [input.propositions[i], reference.propositions[j]];
    rows.push({
      input_seg: inc.seg_id, input: flatten(inc.text),
      reference_seg: ref.seg_id, reference: flatten(ref.text),
      relevance: score.relevance, same_matter: score.sameMatter, differs: score.differs,
      conflict: score.conflict,
      deviating: deviating.has(key),
    });
  }

  rows.sort((a, b) => b.relevance - a.relevance);

  return rows;
}

async function buildReport({
  reference, input, referencePath, inputPath,
  deviationRows, gapsReference, gapsInput,
  inputStatus, referenceStatus, inputFiltered, referenceFiltered, pairs,
}) {
  return {
    schema: SCHEMA,
    reference: await documentSummary(reference, referencePath),
    input: await documentSummary(input, inputPath),
    summary: {
      deviation_rows: deviationRows.length,
      deviations: deviationRows.reduce((sum, row) => sum + row.match_count, 0),
      shortlisted_pairs: pairs.length,
      gaps_reference: gapsReference.length,
      gaps_input: gapsInput.length,
      reference_coverage: coverageCounts(referenceStatus, referenceFiltered),
      input_coverage: coverageCounts(inputStatus, inputFiltered),
    },
    summaries: null,
    recommendations: null,
    deviations: deviationRows,
    gaps_reference: gapsReference,
    gaps_input: gapsInput,
    pairs,
  };
}

function slug(name) {
  return name.toLowerCase().replace(/[^a-z0-9]+/g, "-").replace(/^-+|-+$/g, "").slice(0, 60)
    || "document";
}

function pairSlug(referencePath, inputPath) {
  return `${slug(path.parse(referencePath).name)}__${slug(path.parse(inputPath).name)}`;
}

function analysisParams() {
  return {
    schema: SCHEMA,
    pipeline_version: PIPELINE_VERSION,
    enricher_model: ENRICHER_MODEL,
    embedder_model: EMBEDDER_MODEL,
    classifier_model: CLASSIFIER_MODEL,
    shortlist_min_relevance: SHORTLIST_MIN_RELEVANCE,
    shortlist_top_k: SHORTLIST_TOP_K,
    filter_query: FILTER_QUERY,
    filter_threshold: FILTER_THRESHOLD,
    reranker_model: RERANKER_MODEL,
    empty_query: EMPTY_QUERY,
    care_statement: CARE_STATEMENT,
    agree_query: AGREE_QUERY,
    money_statement: MONEY_STATEMENT,
    classify_query_max_chars: CLASSIFY_QUERY_MAX_CHARS,
    match_temperature: MATCH_TEMPERATURE,
    same_slope: SAME_SLOPE,
    same_bias: SAME_BIAS,
    same_threshold: SAME_THRESHOLD,
    materiality_weights: [EMPTY_WEIGHT, CARE_WEIGHT, AGREE_WEIGHT, MONEY_WEIGHT],
    differs_slope: DIFFERS_SLOPE,
    differs_bias: DIFFERS_BIAS,
    conflict_slopes: [CONFLICT_SAME_SLOPE, CONFLICT_DIFFERS_SLOPE],
    conflict_bias: CONFLICT_BIAS,
    conflict_threshold: CONFLICT_THRESHOLD,
    structural_categories: [...STRUCTURAL_CATEGORIES],
  };
}

function comparisonDirFor(referencePath, inputPath, { runsDir = REPORTS_DIR } = {}) {
  return path.join(runsDir, pairSlug(referencePath, inputPath));
}

function readJson(file) {
  try {
    return JSON.parse(fs.readFileSync(file, "utf8"));
  } catch {
    return null;
  }
}

async function savedReportIfCurrent(comparisonDir, referencePath, inputPath) {
  const report = readJson(path.join(comparisonDir, "report.json"));
  const saved = readJson(path.join(comparisonDir, "params.json"));

  if (report === null || saved === null) {
    return null;
  }

  const current = analysisParams();

  if (JSON.stringify(Object.fromEntries(Object.keys(current).map((k) => [k, saved[k] ?? null])))
    !== JSON.stringify(current)) {
    return null;
  }

  for (const [side, docPath] of [["reference", referencePath], ["input", inputPath]]) {
    const text = await loadText(docPath);

    if ((report[side] ?? {}).text_sha256 !== crypto.createHash("sha256").update(text, "utf8").digest("hex")) {
      return null;
    }
  }

  return report;
}

async function writeAnalysisRun(report, {
  reference, input, referencePath, inputPath, runDir,
  label = null, documentsDir = DOCUMENTS_DIR,
}) {
  fs.rmSync(runDir, { recursive: true, force: true });
  fs.mkdirSync(runDir, { recursive: true });

  const params = {
    ...analysisParams(),
    slug: path.basename(runDir),
    label,
    created_at: new Date().toISOString().replace(/\.(\d{3})Z$/, ".$1000Z"),
  };

  for (const [name, payload] of [
    ["report", Object.fromEntries(Object.entries(report).filter(([k]) => k !== "pairs"))],
    ["pairs", report.pairs || []],
    ["params", params],
  ]) {
    fs.writeFileSync(path.join(runDir, `${name}.json`), JSON.stringify(payload, null, 2), "utf8");
  }

  await writeDocumentArtifact(referencePath, reference, { documentsDir });
  await writeDocumentArtifact(inputPath, input, { documentsDir });

  return runDir;
}

2. Add the pipeline. `runComparison()` runs every stage in order for one pair, and `useCache: false` runs everything from scratch. The `STAGE:` lines follow the overview, with Report split into assembling and saving.

In [ ]:
async function runComparison(client, referencePath, inputPath, { useCache = true, label = null } = {}) {
  const dest = comparisonDirFor(referencePath, inputPath);
  const saved = useCache ? await savedReportIfCurrent(dest, referencePath, inputPath) : null;

  if (saved !== null) {
    return saved;
  }

  console.log("STAGE: Reading documents");
  const reference = await indexDocument(client, referencePath, { useCache });
  const input = await indexDocument(client, inputPath, { useCache });

  console.log("STAGE: Filtering");
  const [inputFiltered, referenceFiltered] = [await nonOperative(client, input), await nonOperative(client, reference)];

  console.log("STAGE: Matching");
  const shortlist = shortlistPairs(
    maskNonOperative(pairMatrix(input, reference), inputFiltered, referenceFiltered),
    { minRelevance: SHORTLIST_MIN_RELEVANCE, topK: SHORTLIST_TOP_K },
  );

  console.log("STAGE: Classifying");
  const scored = await scorePairs(client, input, reference, { shortlist });

  console.log("STAGE: Comparing");
  const [deviatingPairs, inputStatus, referenceStatus] = partition(input, reference, scored);

  const deviationRows = rollUpDeviations(deviatingPairs, input, reference);
  const gapsReference = rollUpGaps(reference, referenceStatus, { other: input, filtered: referenceFiltered });
  const gapsInput = rollUpGaps(input, inputStatus, { other: reference, filtered: inputFiltered });

  console.log("STAGE: Assembling report");
  const report = await buildReport({
    reference, input, referencePath, inputPath,
    deviationRows, gapsReference, gapsInput,
    inputStatus, referenceStatus, inputFiltered, referenceFiltered,
    pairs: scoredPairsRows(
      input, reference, scored, { deviating: new Set(deviatingPairs.map(([i, j]) => `${i}:${j}`)) },
    ),
  });

  console.log("STAGE: Saving results");
  await writeAnalysisRun(report, {
    reference, input, referencePath, inputPath, runDir: dest, label,
  });

  return report;
}

3. Run it against the sample pair. This is the first time the whole pipeline runs end to end. The report's `summaries` is `null` for now: part C fills it in.

In [ ]:
const report = await runComparison(
  client,
  path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md"),
  path.join(SAMPLES_DIR, "04-mistral-ai-terms-of-use.md"),
);

Expected output:

```text
STAGE: Reading documents
STAGE: Filtering
STAGE: Matching
STAGE: Classifying
STAGE: Comparing
STAGE: Assembling report
STAGE: Saving results
```

- We get a line per stage, then a new folder under `results/reports/` holding `report.json`, `pairs.json` and `params.json`, and a record of each document under `results/documents/`.

- 56 deviations in 54 rows (two input clauses have two counterparts each), and many more gaps, is about right for two different agreements that overlap only in part.
- Hundreds of deviations on a pair that should mostly agree means `CONFLICT_THRESHOLD` is too low; none on a pair you know disagrees means it's too high.
- Read the pairs, not the count. Each deviation's `matches` in `report.json` should say different things about the same point; expect a few that only rename a defined term.
- Each row's `heading` should be a clause a reviewer could find in the document.
- Your counts may differ by one or two, and scores by a hundredth or so, between runs.

Swap in any other document from `samples/` to check a new pair. Running the same pair again with nothing changed reuses the saved run and costs nothing; pass `useCache: false` to start over.

## C. Generation (optional)

We can make our lives easier by having a generative model summarize each finding and make recommendations for us. _Document analysis_ creates a saved analysis report. This optional step adds summaries and recommendations to that same report file.

Here's a deviation from the sample pair:

- **Cloud Service Agreement (reference):** "Except as provided in Section 8.4 (Exceptions), each party's total cumulative liability for all claims arising out of or relating to this Agreement will not be more than the General Cap Amount."
- **Mistral AI Terms of Use (input):** "To the extent permitted by law, the total aggregate liability of Mistral AI in respect of any Losses incurred by the Customer under or in relation to this Agreement will not exceed, in the aggregate, the lower of (i) the amount of the Fees paid or payable by the Customer in the twelve (12) calendar months preceding the date on which the first such event or events occurred or (ii) 10.000 euros."

$P(\text{same})$ of 0.68 puts them on the same matter; with $P(\text{differs})$ of 0.99, $P(\text{conflict})$ is 0.81, over `CONFLICT_THRESHOLD`.

Our optional third part, _generation_, turns those results into something more user-friendly. It's easier to read a summary of what the values mean than to decipher the meaning after reading two pairs of provisions. These models are generative, so a watchful eye is certainly needed in case of hallucination and other errors.

![Matt Damon reacting in The Odyssey](https://isaacus.com/blog/build-a-legal-document-conflict-detector/the-odyssey-matt-damon.gif)

_Don't fall for generative shenanigans._

I've found generative models giving me different results even with similar prompts, so it's worth trying a few for preferences' sake. I was working with Opus and Sonnet; you should customize the prompts for models you prefer to use.

The code for _generation_ is long because it supports several LLM providers and adds to the report. I'll summarize the quirks of the prompts instead.

There are two prompts. Their placeholders are filled with the documents, parties, and findings at run time.

`PARTIES_PROMPT` names the parties once per pair, from the input document only. Legislation or a policy has no signing parties, so it lists the groups the document binds or protects instead, such as "employers" and "employees".

Expected `PARTIES_PROMPT`:

```text
Below is an input document. Identify its major parties: the people or organisations that sign it or are bound by it as a party. Leave out anyone who is only mentioned, such as affiliates, third parties or regulators. If the input document is not an agreement (for example, legislation or a policy), list instead the main groups it binds or protects, such as "employers" and "employees".

Respond with ONLY a JSON array, no prose and no markdown code fences, of {"id": short lowercase slug, "name": the name as the input document gives it (e.g. "The Walt Disney Company", "Robert A. Iger", or a role like "the Customer" if the input document names no one), "role": a few words on who they are in this deal}.

<input_document>
{input}
</input_document>
```

`RECOMMEND_PROMPT` takes the findings in batches. For each one, it writes the summaries, a comment, an action and reason for each party, and a rewrite if appropriate. Some of the quirks:

- It writes a finding's summaries, comment, actions and rewrite in one answer, so they don't contradict each other.
- Every call includes both full documents. When summaries were written from the two passages alone, they often said the other document lacked something that was in a different clause.
- The model can't recommend `remove` for a missing clause, or for a passage that another clause refers to. Before this rule, it deleted subclauses that other clauses cite by number.
- A rewrite is only a suggestion until the reviewer accepts it, so the model judges every finding against the input document as written. A rewrite goes no further than the reference position, and keeps the input document's numbering, list markers and line breaks so the redline shows only the real change. An added passage can get no rewrite (`null`) when the reference document has no such passage, so the model doesn't invent a narrower clause.

Expected `RECOMMEND_PROMPT`:

```text
You are a senior lawyer. A client checks an input document against its standard, the reference document. For each finding in <findings>, write summaries, a comment, a recommendation for each party to the input document and a rewrite. <all_findings> lists every finding in this review as context; several can be on the same point.

Read both documents in full first. A finding is what a detector matched, not a confirmed difference or absence, so check it against both documents. Read qualifiers exactly ("to the extent", "unless"). Never describe a limited rule as unlimited.

Parties to the input document:
{parties}

Kinds of finding: "changed" (an input passage matched with one or more reference passages that may differ from it), "added" (an input passage that matched no reference passage) and "missing" (a reference passage that matched no input passage).

Judge each finding against the input document as it is now, never against a rewrite that you give for another finding.

Fields:
- "summaries": for a reader who skims the findings. One or two sentences each, 40 words at most. For "changed", give one per reference passage, in order: start with "The input document", then say what the reference document does instead, with the exact amount, date, condition or party that differs and its qualifiers. If only the wording differs, write exactly "Same in substance; only the wording differs." instead. For "added" and "missing", give one: what the passage secures and for which party, with its qualifiers. Never say what happens without it or that another clause lacks it.
- "falsePositive": true only if, with both full documents in view, no part of the finding changes the effect. One equivalent reference passage does not make the whole finding false. A different law, price, amount or date is a real difference.
- "comment": two to four sentences: what the input document does, what the reference document does instead, and which party this favours and why. Cite a reference clause by the number and heading it has, for example "Section 4.1 (Fees) of the reference document", after you check them.
- "referenceQuotes": exact excerpts, 80 words at most each, from other reference passages that the comment relies on. [] if none.
- "parties": for each party id, from that party's point of view:
  - "action", in this order: "ignore" if the finding is a false positive; null if it is a commercial choice, such as a price, salary or date; else "rewrite" if the rewrite helps the party, "remove" if the party gains most when the whole input passage goes (not for "missing", nor when another clause refers to it), or "ignore".
  - "reason": one sentence on what the party gains or loses, in concrete terms. Name another input clause only if it already gives the same result.
- "rewrite": the passage at the reference document's position. For "changed", redraft the input passage. For "added", narrow or qualify it toward the reference position, or give null if that position is to have no such passage. For "missing", draft the clause to insert, and cover only this finding's reference passage.
- "insertAfter": for "missing", copy exactly the end of the input document paragraph that the new clause follows: its last 8 to 15 words, or all of it if it is shorter, with its punctuation and Markdown. These words must occur only once in the input document. Choose the place a lawyer would choose: in the right section, after the most related clause. Else null.

Summaries, comment and reasons:
- Write ASD-STE100 Simplified Technical English: active voice, one idea per sentence, no sentence over 20 words, common words with one meaning.
- Use plain words for legal ideas: "a chance to fix the problem first", not "a cure period". Never write "shall", "herein", "pursuant to", "notwithstanding", "the foregoing" or "in respect of".
- Call the documents "the input document" and "the reference document", never by title or year. Name each party, for example "the Customer". Never "You" or "We".
- Never mention finding ids, the detector, this review or the rewrite. Plain text, no Markdown.
- They must agree with each other. Check each of them against the exact words of the passages before you answer.

Rewrite:
- Write it as part of the input document, in its drafting style: its defined terms, party names, capitalisation, legal words and cross-reference style, in Markdown. Adapt the reference wording to the input document. Never cite a reference section number or use a term the input document does not define.
- Go no further than the reference position. Never limit or widen an obligation the finding is not about.
- "changed" and "added": change only what the finding is about. Keep all other words, numbering, list markers and line structure.
- "missing": write the whole clause, numbered to fit its place.
- The rewrite must not contradict or repeat another clause. If it must override a clause that this finding cannot change, start with "Despite Section X,". After you move or add words, check that "this" and "such" still point to the right thing.
- Where a rule depends on who acts, say who.

Respond with ONLY one JSON object, no prose and no code fences. Map each finding id in <findings>, exactly as given, to:
{"summaries": [string], "falsePositive": boolean, "comment": string, "referenceQuotes": [string], "parties": {<party id>: {"action": "ignore" | "remove" | "rewrite" | null, "reason": string}}, "rewrite": string (null only for "added"), "insertAfter": string or null}
Include each of those finding ids once, and every party id in each.

<reference_document>
{reference}
</reference_document>

<input_document>
{input}
</input_document>

<all_findings>
{index}
</all_findings>

<findings>
{findings}
</findings>
```

1. You can have a look at the generation file ([Python](https://github.com/isaacus-dev/cookbooks/blob/main/cookbooks/conflict-detector/source_files/python/generation.py) or [JavaScript](https://github.com/isaacus-dev/cookbooks/blob/main/cookbooks/conflict-detector/source_files/js/generation.js)) containing the prompts above if you like. Import it and run it on the pair from _document analysis_. Set `model` to the model you want, and `api_key` to its key. The default, `claude-cli-opus`, runs the Claude CLI on your Claude Code sign-in, so it needs no key. For an API, use a `claude-`, `gpt-` or `gemini-` model ID. `generateRun()` reads the pair's `report.json`, fills in `summaries` and `recommendations` and saves it again. It keeps findings already saved for the same model; pass `useCache: false` to write them again.

In [ ]:
import * as generation from "./source_files/js/generation.js";

await generation.generateRun(
  comparisonDirFor(
    path.join(SAMPLES_DIR, "03-common-paper-cloud-service-agreement.md"),
    path.join(SAMPLES_DIR, "04-mistral-ai-terms-of-use.md"),
  ),
  { model: "claude-cli-opus", apiKey: null },
);

Expected `stage messages (batch progress omitted)`:

```text
STAGE: Finding parties
STAGE: Summarizing and recommending
```

### The results

`report.json` has a `summaries` block. Here's an example, trimmed to the liability cap deviation and the first gap on the reference side:

Expected `report.json`:

```json
{
  "model": "claude-cli-opus",
  "matches": [
    {
      "key": "d36-0",
      "deviation_id": 36,
      "match_index": 0,
      "text": "The input document caps only Mistral AI's liability, as far as the law permits. The cap is the lower of 12 months' Fees or 10,000 euros. The reference document caps each party at the General Cap Amount, with stated exceptions."
    }
  ],
  "gaps": [
    {
      "key": "r1",
      "gap_id": 1,
      "side": "reference",
      "text": "The reference document lets the Customer use the Cloud Service, Software and Documentation during the Subscription Period, for internal business purposes only. An Affiliate's separate Order Form makes a separate agreement, and the Customer is not responsible for it."
    }
  ]
}
```

You can find this in the demo above.

### Recommendations

Here is the `recommendations` block from the same run, trimmed to the liability cap finding:

Expected `report.json`:

```json
{
  "model": "claude-cli-opus",
  "parties": [
    {
      "id": "mistral-ai",
      "name": "Mistral AI",
      "role": "provider of the AI models and platform services"
    },
    {
      "id": "customer",
      "name": "the Customer",
      "role": "user or subscriber of the services"
    }
  ],
  "findings": {
    "d36-0": {
      "falsePositive": false,
      "comment": "The input document caps only Mistral AI's liability for the Paid Services, as far as the law permits. The cap is the lower of 12 months' Fees or 10,000 euros. Section 8.1 (Liability Caps) of the reference document caps each party at the General Cap Amount, subject to Section 8.4 (Exceptions). The input document favours Mistral AI, because the liability of the Customer has no cap.",
      "rewrite": "**Liability Cap for Paid Services**. To the extent permitted by law, the total aggregate liability of each Party in respect of any Losses incurred by the other Party under or in relation to this Agreement will not exceed, in the aggregate, the lower of (i) the amount of the Fees paid or payable by the Customer in the twelve (12) calendar months preceding the date on which the first such event or events occurred or (ii) 10.000 euros.",
      "parties": {
        "mistral-ai": {
          "action": "ignore",
          "reason": "Mistral AI keeps its own cap but could recover only a capped amount from the Customer."
        },
        "customer": {
          "action": "rewrite",
          "reason": "The Customer's liability for the Paid Services would also stop at the lower of 12 months' Fees or 10,000 euros."
        }
      },
      "references": [
        {
          "start": 14274,
          "end": 14418
        }
      ]
    }
  }
}
```

## Further development

You've built a conflict checker containing normalization, indexing, and analysis that can check one document against another and write down every deviation and gap. No vector database, and no generative model deciding what counts as a conflict. Generation optionally adds a written summary and recommendation to each finding.

There are plenty of ways to extend it:

- **Tune it.** The questions, the weights and the thresholds can be refitted to your own reference documents; `pairs.json` keeps every answer, so you can try new weights or a new threshold against a saved run for free. A lower `CONFLICT_THRESHOLD` finds more real differences at the cost of more false alarms.
- **Mind your inputs.** The pipeline expects clean markdown. Plain text gets converted for you, but if you've got tables or scanned documents, fix the files rather than adding rules for them.
- **Run your reference documents against each other.** Find the conflicts in your own templates (not some meta conflictception, I promise).
- **Make it faster.** Cache classifier results, skip repeated clause pairs, and find the sweet spot for parallel requests. Originally I'd kept a full history of results plus a dashboard, but, y'know... Brevity.
- **Build on it.** Redlining, review queues, pre-signing checklists, template and playbook checks, and comparisons when a reference document is updated. It's a demo so I didn't do any of that, but I needed you to know I thought of it. 🩵

If you somehow managed to scroll this far without actually doing any coding, the notebook is available in our [cookbook](https://github.com/isaacus-dev/cookbooks) and the demo files are available separately on GitHub. Run it on your own files – good luck!

![A chaotic battle scene from The Odyssey](https://isaacus.com/blog/build-a-legal-document-conflict-detector/the-odyssey-battle.gif)

_Me getting ready to build more demos for you after this *harrowing* tutorial._